In [2]:
import os
import sys
import logging
from pathlib import Path
import config
import importlib

importlib.reload(config)

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "config.py").exists():
    BASE_DIR = CURRENT_DIR
else:
    BASE_DIR = CURRENT_DIR / "rpi_system"

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import config
import tensorflow as tf
from tensorflow.keras import layers, models

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("Trainer")

# Point to raw-img dataset
DATASET_DIR = config.PROJECT_ROOT / "Data" / "Testing" / "raw-img"

print("TensorFlow Version:", tf.__version__)
print("Dataset Directory :", DATASET_DIR)

TensorFlow Version: 2.20.0
Dataset Directory : C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\Data\Testing\raw-img


In [3]:
# Load Dataset (80% Train, 20% Val)
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=config.IMAGE_SIZE,
    batch_size=config.BATCH_SIZE,
    label_mode="categorical",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=config.IMAGE_SIZE,
    batch_size=config.BATCH_SIZE,
    label_mode="categorical",
)

# Italian-to-English Translation Dictionary
TRANSLATE = {
    "cane": "Dog",
    "cavallo": "Horse",
    "elefante": "Elephant",
    "farfalla": "Butterfly",
    "gallina": "Chicken",
    "gatto": "Cat",
    "lion": "Lion",
    "mucca": "Cow",
    "pecora": "Sheep",
    "scoiattolo": "Squirrel",
}

raw_classes = train_ds.class_names
class_names = [TRANSLATE.get(name.lower(), name.capitalize()) for name in raw_classes]
num_classes = len(class_names)
logger.info(f"Detected {num_classes} classes: {class_names}")

# Save English labels to models/labels.txt
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.OUTPUT_LABELS, "w", encoding="utf-8") as f:
    for name in class_names:
        f.write(f"{name}\n")
logger.info(f"Saved English class labels to: {config.OUTPUT_LABELS}")

# Prefetch for GPU/CPU pipeline speed
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 24033 files belonging to 10 classes.
Using 19227 files for training.
Found 24033 files belonging to 10 classes.
Using 4806 files for validation.


15:07:36 [INFO] Detected 10 classes: ['Dog', 'Horse', 'Elephant', 'Butterfly', 'Chicken', 'Cat', 'Lion', 'Cow', 'Sheep', 'Squirrel']
15:07:36 [INFO] Saved English class labels to: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\rpi_system\models\labels.txt


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name="data_augmentation")

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(config.IMAGE_SIZE[0], config.IMAGE_SIZE[1], 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # Freeze ImageNet backbone

inputs = tf.keras.Input(shape=(config.IMAGE_SIZE[0], config.IMAGE_SIZE[1], 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="edge_ai_animal_detector")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

Model: "edge_ai_animal_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 7, 7, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 576)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         5,770 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 944,890 (3.60 MB)

 Trainable params: 5,770 (22.54 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [5]:
logger.info(f"Training for up to {config.EPOCHS} epochs...")
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=config.EPOCHS,
    callbacks=callbacks,
)

val_loss, val_acc = model.evaluate(val_ds)
logger.info(f"Training Complete! Final Validation Accuracy: {val_acc:.2%}")

15:07:37 [INFO] Training for up to 30 epochs...


Epoch 1/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 146s 470ms/step - accuracy: 0.7379 - loss: 0.8250 - val_accuracy: 0.9037 - val_loss: 0.3292 - learning_rate: 0.0010
Epoch 2/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 143s 477ms/step - accuracy: 0.8574 - loss: 0.4420 - val_accuracy: 0.9166 - val_loss: 0.2640 - learning_rate: 0.0010
Epoch 3/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 141s 470ms/step - accuracy: 0.8786 - loss: 0.3757 - val_accuracy: 0.9230 - val_loss: 0.2342 - learning_rate: 0.0010
Epoch 4/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 149s 496ms/step - accuracy: 0.8843 - loss: 0.3582 - val_accuracy: 0.9257 - val_loss: 0.2225 - learning_rate: 0.0010
Epoch 5/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 140s 464ms/step - accuracy: 0.8896 - loss: 0.3369 - val_accuracy: 0.9303 - val_loss: 0.2114 - learning_rate: 0.0010
Epoch 6/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 139s 460ms/step - accuracy: 0.8908 - loss: 0.3319 - val_accuracy: 0.9288 - val_loss: 0.2149 - learning_rate: 0.0010
Epoch 7/30
301/301 ━━━━━━━━━━━━━━━━━━━━ 135s 448ms/step - accura

15:38:49 [INFO] Training Complete! Final Validation Accuracy: 93.55%


In [6]:
# PHASE 2: FINE-TUNING (UNFREEZE UPPER CONVOLUTIONAL LAYERS)
logger.info("Starting Phase 2: Fine-Tuning...")

# Unfreeze the base model
base_model.trainable = True

# Keep early layers (basic edges/colors) frozen, adapt complex shape layers
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

logger.info(f"Unfrozen {len(base_model.layers) - fine_tune_at} / {len(base_model.layers)} layers for fine-tuning.")

# Recompile with a smaller learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Fine-tune callbacks (patience=3)
fine_tune_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

# Resume training for another 10-15 epochs
fine_tune_epochs = 15
total_epochs = history.epoch[-1] + 1 + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=history.epoch[-1] + 1,
    epochs=total_epochs,
    callbacks=fine_tune_callbacks,
)

# Evaluate final fine-tuned accuracy
val_loss, val_acc = model.evaluate(val_ds)
logger.info(f"Fine-Tuning Complete! Final Out-of-Sample Accuracy: {val_acc:.2%}")


15:38:49 [INFO] Starting Phase 2: Fine-Tuning...
15:38:49 [INFO] Unfrozen 30 / 157 layers for fine-tuning.


Epoch 14/28
301/301 ━━━━━━━━━━━━━━━━━━━━ 168s 536ms/step - accuracy: 0.8608 - loss: 0.4231 - val_accuracy: 0.9322 - val_loss: 0.1997 - learning_rate: 1.0000e-05
Epoch 15/28
301/301 ━━━━━━━━━━━━━━━━━━━━ 161s 534ms/step - accuracy: 0.8831 - loss: 0.3595 - val_accuracy: 0.9328 - val_loss: 0.2012 - learning_rate: 1.0000e-05
Epoch 16/28
301/301 ━━━━━━━━━━━━━━━━━━━━ 0s 493ms/step - accuracy: 0.8880 - loss: 0.3393
Epoch 16: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
301/301 ━━━━━━━━━━━━━━━━━━━━ 175s 581ms/step - accuracy: 0.8913 - loss: 0.3348 - val_accuracy: 0.9324 - val_loss: 0.1999 - learning_rate: 1.0000e-05
Epoch 17/28
301/301 ━━━━━━━━━━━━━━━━━━━━ 166s 550ms/step - accuracy: 0.8913 - loss: 0.3292 - val_accuracy: 0.9340 - val_loss: 0.1984 - learning_rate: 2.0000e-06
Epoch 18/28
301/301 ━━━━━━━━━━━━━━━━━━━━ 156s 518ms/step - accuracy: 0.8905 - loss: 0.3321 - val_accuracy: 0.9340 - val_loss: 0.1982 - learning_rate: 2.0000e-06
Epoch 19/28
301/301 ━━━━━━━━━━━━━━━━━━━━

16:08:49 [INFO] Fine-Tuning Complete! Final Out-of-Sample Accuracy: 93.51%


In [7]:
# 1. Setup the Converter
logger.info("Converting model to FULL INT8 .tflite for Raspberry Pi...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 2. Provide a "Representative Dataset" (a small sample of training data)
# This is REQUIRED for Full INT8 quantization.
def representative_data_gen():
    # Take 100 batches from the training dataset
    for input_value, _ in train_ds.take(100):
        yield [input_value]

converter.representative_dataset = representative_data_gen

# 3. Force the model to be STRICTLY INT8 operations (no fallback to float)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8  # Makes input take standard image bytes
converter.inference_output_type = tf.uint8 # Makes output standard bytes

# 4. Convert and Save
tflite_model = converter.convert()

with open(config.OUTPUT_TFLITE, "wb") as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / (1024 * 1024)
logger.info(f"Saved Full INT8 TFLite model to: {config.OUTPUT_TFLITE} ({size_mb:.2f} MB)")

16:08:49 [INFO] Converting model to FULL INT8 .tflite for Raspberry Pi...
16:08:50 [INFO] Function `function` contains input name(s) resource with unsupported characters which will be renamed to edge_ai_animal_detector_1_dense_1_biasadd_readvariableop_resource in the SavedModel.
16:08:50 [INFO] Function `function` contains input name(s) resource with unsupported characters which will be renamed to edge_ai_animal_detector_1_dense_1_biasadd_readvariableop_resource in the SavedModel.


INFO:tensorflow:Assets written to: C:\Users\sabbu\AppData\Local\Temp\tmp87gjag8c\assets


16:08:53 [INFO] Assets written to: C:\Users\sabbu\AppData\Local\Temp\tmp87gjag8c\assets


Saved artifact at 'C:\Users\sabbu\AppData\Local\Temp\tmp87gjag8c'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_175')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  2815288552400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288554128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288553936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288553552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288554704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288553168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288554320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288554512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288551440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2815288555664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  

C:\Users\sabbu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\lite\python\convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
16:10:55 [INFO] Saved Full INT8 TFLite model to: C:\Users\sabbu\OneDrive\Documents\Git\Edge-Ai-CNN-detection\rpi_system\models\animal_classifier.tflite (1.17 MB)


In [10]:
import cv2
import numpy as np

interpreter = tf.lite.Interpreter(model_path=str(config.OUTPUT_TFLITE))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

with open(config.OUTPUT_LABELS, "r") as f:
    labels = [line.strip() for line in f if line.strip()]

# Test on a sample cat image from the dataset
test_img_path = next((DATASET_DIR / "gatto").glob("*.jpeg"))
print(f"Testing on: {test_img_path.name}")

img = cv2.imread(str(test_img_path))
rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
resized = cv2.resize(rgb_img, (config.INPUT_WIDTH, config.INPUT_HEIGHT))
input_data = np.expand_dims(resized, axis=0).astype(np.uint8)

interpreter.set_tensor(input_details[0]["index"], input_data)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])[0]

top_idx = int(np.argmax(output))
print(f"Predicted Animal : {labels[top_idx]}")
print(f"Confidence Score : {output[top_idx] * 100:.2f}%")

Testing on: 1.jpeg
Predicted Animal : Cat
Confidence Score : 44.00%


C:\Users\sabbu\AppData\Local\Temp\ipykernel_7052\2074203215.py:27: RuntimeWarning: overflow encountered in scalar multiply
  print(f"Confidence Score : {output[top_idx] * 100:.2f}%")
